In [118]:
import pandas as pd

df = pd.read_excel("master_data.xlsx")

df.fillna("", inplace=True)

In [135]:
import pandas as pd


# =========================
# LOAD ALL EXCEL SHEETS
# =========================

all_sheets = pd.read_excel(
    "master_data.xlsx",
    sheet_name=None
)

# combine all sheets
df = pd.concat(
    all_sheets.values(),
    ignore_index=True
)

# fill missing values
df = df.fillna("")


# =========================
# CLEANING
# =========================

# lowercase column names
df.columns = df.columns.str.lower()

# columns to clean
columns_to_clean = [
    'sector',
    'beneficiary type',
    'women_focused',
    'central/state',
    'target region/state'
]

# clean only existing columns
for col in columns_to_clean:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .str.strip()
        )


# =========================
# RECOMMENDATION FUNCTION
# =========================

def recommend_schemes_scored(
    sector=None,
    beneficiary=None,
    women_focused=None,
    state=None,
    central_state=None,
    top_n=20
):

    # create copy
    filtered_df = df.copy()


    # =========================
    # CLEAN USER INPUTS
    # =========================

    if sector:
        sector = sector.lower().strip()

    if beneficiary:
        beneficiary = beneficiary.lower().strip()

    if women_focused:
        women_focused = women_focused.lower().strip()

    if state:
        state = state.lower().strip()

    if central_state:
        central_state = central_state.lower().strip()


    # =========================
    # CREATE SCORE COLUMN
    # =========================

    filtered_df['score'] = 0


    # =========================
    # SECTOR SCORING
    # =========================

    if sector and 'sector' in filtered_df.columns:

        filtered_df.loc[
            filtered_df['sector']
            .str.contains(sector, na=False),

            'score'
        ] += 3


    # =========================
    # BENEFICIARY SCORING
    # =========================

    if beneficiary and 'beneficiary type' in filtered_df.columns:

        filtered_df.loc[
            filtered_df['beneficiary type']
            .str.contains(beneficiary, na=False),

            'score'
        ] += 2


    # =========================
    # WOMEN FOCUSED SCORING
    # =========================

    if women_focused and 'women_focused' in filtered_df.columns:

        filtered_df.loc[
            filtered_df['women_focused']
            == women_focused,

            'score'
        ] += 1


    # =========================
    # STATE FILTERING + SCORING
    # =========================

    if state and 'target region/state' in filtered_df.columns:

        # keep only state + pan india + all india
        filtered_df = filtered_df[

            filtered_df['target region/state']
            .str.contains(state, na=False)

            |

            filtered_df['target region/state']
            .str.contains('pan india', na=False)

            |

            filtered_df['target region/state']
            .str.contains('all india', na=False)

        ]


        # extra score for exact state match
        filtered_df.loc[
            filtered_df['target region/state']
            .str.contains(state, na=False),

            'score'
        ] += 2


    # =========================
    # CENTRAL / STATE SCORING
    # =========================

    if central_state and 'central/state' in filtered_df.columns:

        filtered_df.loc[
            filtered_df['central/state']
            == central_state,

            'score'
        ] += 2


    # =========================
    # REMOVE ZERO SCORE ROWS
    # =========================

    filtered_df = filtered_df[
        filtered_df['score'] > 0
    ]


    # =========================
    # SORT BY SCORE
    # =========================

    filtered_df = filtered_df.sort_values(
        by='score',
        ascending=False
    )


    # =========================
    # IMPORTANT OUTPUT COLUMNS
    # =========================

    useful_columns = [
        'scheme name',
        'sector',
        'beneficiary type',
        'women_focused',
        'target region/state',
        'central/state',
        'subsidy/grant support',
        'score'
    ]

    existing_columns = [
        col for col in useful_columns
        if col in filtered_df.columns
    ]


    # =========================
    # RETURN TOP RESULTS
    # =========================

    return filtered_df[
        existing_columns
    ].head(top_n)



# =========================
# TEST
# =========================

recommend_schemes_scored(
    sector="fpo",
    women_focused="yes",
    state="uttar pradesh",
    central_state="state"
)

,scheme name,sector,beneficiary type,women_focused,target region/state,central/state,subsidy/grant support,score
1091,UP FPO Grant Scheme (Drishti Scheme),fpo support,farmer producer organizations (fpos),not mentioned,uttar pradesh,state,Grant of ₹60 lakh per FPO unit for 100 FPOs,7
2233,One District One Product (ODOP) Initiative,artisan & handicraft infrastructure,artisans; shgs; women; youth; traditional craf...,yes,uttar pradesh,state,Vishwakarma Shramik Samman; modern toolkits di...,5
1092,"Central FPO Formation Scheme (10,000 FPOs) - U...",fpo support,"small and marginal farmers, fpos",not mentioned,uttar pradesh,central,Management financial assistance up to ₹18 lakh...,5
221,Resham Sakhi Yojana,sericulture,rural women,yes,uttar pradesh (15 districts in first phase),state,"Training provided; aims to connect 50,000 wome...",5
1090,FPO/Cooperative Oil Extraction Unit Subsidy (N...,fpo support,"registered fpos, cooperative societies",not mentioned,uttar pradesh,central,Grant of ₹9.90 lakh per unit (33% of project c...,5
2235,Micro-Enterprise Sakhis Programme - State Leve...,shg enterprise infrastructure,"rural women (13,064 women to be trained as sak...",yes,uttar pradesh,state,Training for Micro-Enterprise Sakhis; Honorari...,5
230,Intensive Fish Farming Aeration System Scheme,fisheries,women fish farmers,yes,uttar pradesh,state,"50% subsidy on unit cost of ₹50,000 (exclusive...",5
1052,"Formation and Promotion of 10,000 Farmer Produ...",fpo support,"small and marginal farmers, fpos, women, sc, st",yes,pan india,central,Financial assistance up to ₹18.00 lakh per FPO...,4
229,Nishadraj Boat Subsidy Scheme,fisheries,traditional fishermen/boat owners,not mentioned,uttar pradesh,state,"40% subsidy on unit cost of ₹77,050",4
1695,UPSIDA Land Allotment for Food Processing Indu...,food processing infrastructure,"food processing companies, agri-processing ent...",not mentioned,uttar pradesh,state,Land allotment at industrial areas; Investment...,4
